# Experimento — MLP, fase 4: medir melhor antes de escolher melhor

A fase 3 mostrou dois problemas de avaliação. A **parada pelo KS** ganhou na validação em 20 de 20 divisões e não mudou nada no teste, porque a mesma dobra decidia a parada e media o KS. E a linhagem escolhida pelo teste chegou a 0,5655, mas perdia 0,022 quando medida em clientes que não participaram da escolha. Nenhuma das duas era uma melhora de verdade.

Esta fase troca a régua antes de procurar modelos melhores.

## A medida nova: k-fold do treino com separação interna

Em cada uma das 5 dobras externas:

1. O resto do treino é dividido de novo: **80% ajustam a rede e 20% decidem a parada**. A dobra externa não participa.
2. A rede é **retreinada em todo o resto**, pela mediana das épocas da parada, como na avaliação final.
3. A dobra externa mede as duas redes. `ks` (retreino) é a medida principal. `ks_parada` mostra a rede que parou.

O scaler, o oversampling e as features criadas são ajustados só na parte que treina. **O teste não entra em nenhuma escolha:** aparece uma única vez, na avaliação final, para o campeão e a referência.

## O que a fase faz

1. **Conferência e protocolos:** a pilha mede igual a `modelo.py`? Quanto o protocolo antigo inflava o KS?
2. **Mapa dos clientes ambíguos:** os clientes perto do limiar do KS, o que ainda os separa e regras que se repetem entre dobras.
3. **Blocos:** ponto de partida (os padrões da fase 3), balanceamento, features para os ambíguos (no MLP **e no XGBoost**) e treino.
4. **Optuna** na medida nova, com o balanceamento dentro da busca.
5. **Confirmação** em 30 divisões que nenhuma escolha viu: a estimativa honesta do ganho.
6. **Avaliação final** no teste, uma vez só.

Cada bloco segue a regra de sempre: ganho médio acima da régua de ruído e vitória em pelo menos 80% das divisões. As divisões de cada etapa são separadas:

| Etapa | Divisões |
|---|---|
| Mapa dos ambíguos | 501 a 510 |
| Blocos | 101 a 130 |
| Optuna | 201 e 202 |
| Aceite do Optuna | 301 a 330 |
| Confirmação | 401 a 430 |
| Comparação de protocolos | 601 a 610 |

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase4.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Na fase 3 o token estava vencido e nada foi enviado ao GitHub.** Gere um token novo (*Fine-grained*, repositório `churn_project`, **Contents: Read and write**) e troque o Secret. A primeira célula de treino testa o envio e avisa se falhar. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp-diagnostico"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase4-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
FASE2 = "resultados/mlp-fase2-20261104-0505"   # a referência: o vencedor da fase 2
USAR_GPU = True                  # treina em pilha na(s) GPU(s); sem GPU, uma rede por vez em vários processos
DISPOSITIVO = "cpu"              # onde roda modelo.py (conferência, Optuna e final): uma rede pequena por vez rende mais na CPU
N_PROCESSOS = 0                  # 0 = todos os núcleos
REPS_CONFERENCIA = 4             # divisões por configuração na conferência entre a pilha e modelo.py
REPS_PROTOCOLO = 10              # divisões na comparação entre o protocolo antigo e o novo
REPS_MAPA = 10                   # divisões para o escore fora da dobra que define os clientes ambíguos
REPS_BLOCO = 30                  # divisões por variante nos blocos
RODADAS_FEATURES = 3             # rodadas do bloco de features (cada rodada parte do que a anterior aceitou)
N_TRIALS = 300                   # trials do Optuna
REPS_CONFIRMACAO = 30            # divisões novas para a estimativa final do ganho
FRACAO_BANDA = 0.3               # fração dos clientes, os mais perto do limiar do KS, tratada como ambígua
N_SEMENTES = 5                   # sementes da avaliação final
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa etapas novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp"), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase4-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if DISPOSITIVO:
    os.environ["CHURN_DEVICE"] = DISPOSITIVO
N_PROCESSOS = N_PROCESSOS or os.cpu_count() or 1
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"
    REPS_CONFERENCIA, REPS_PROTOCOLO, REPS_MAPA, REPS_BLOCO, RODADAS_FEATURES, N_TRIALS, REPS_CONFIRMACAO, N_SEMENTES = 2, 2, 2, 2, 1, 4, 2, 2
    print("MODO_TESTE: 3 épocas e poucas divisões. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


_ultimo_backup = [time.time()]


def backup_periodico():
    if time.time() - _ultimo_backup[0] > 30 * 60:
        salvar(); _ultimo_backup[0] = time.time()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import comum, modelo as M, blocos as BL, fase3 as F3, fase4 as F4, pilha as P
import torch

GPUS = P.dispositivos_disponiveis() if USAR_GPU and torch.cuda.is_available() else None
F4.usar_pilha(GPUS)
F4.abrir_cache(PASTA)
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | pilha: {GPUS or 'desligada'} | modelo.py: {M.DEVICE} | processos: {N_PROCESSOS}")

# O envio ao GitHub falhou a fase 3 inteira sem ninguém perceber: aqui ele é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=4, processos=N_PROCESSOS, gpus=GPUS, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def logar(**k):
    if wandb_run is not None:
        wandb_run.log(k)


LIMITE_VIES = 0.005  # a conferência interrompe a execução acima disto
REFERENCIA = json.load(open(os.path.join(REPO_DIR, FASE2, "blocos", "b12_arquitetura.json")))["campeao"]
REFERENCIA = dict(hp={**REFERENCIA["hp"], "plano": "N"}, dados=REFERENCIA["dados"], extras=REFERENCIA["extras"])
# Os campeões da fase 3, escritos aqui para o notebook não depender da pasta de resultados dela
FASE3_VAL = {'hp': {'camadas': 3, 'unidades': 16, 'ativ': 'gelu', 'otim': 'adam', 'lr': 0.001, 'dropout': 0.2, 'l2': 1e-06, 'perda': 'ce', 'paciencia': 10, 'plano': 'DND', 'parada': 'ks'}, 'dados': {}, 'extras': ['cliente_novo', 'delta_preco']}
FASE3_TESTE = {'hp': {'camadas': 2, 'unidades': 47, 'ativ': 'relu', 'otim': 'adam', 'lr': 0.0001564570789442291, 'dropout': 0.3215897831163661, 'l2': 1.1107958156042701e-06, 'perda': 'mse', 'paciencia': 10, 'plano': 'DD', 'peso_pos': 1.5, 'funil': True, 'suav': 0.0898753403823129, 'pen_logit': 0.0, 'lote': 512, 'parada': 'perda'}, 'dados': {'over': False}, 'extras': ['cliente_novo', 'delta_preco']}
print("Referência (vencedor da fase 2):", json.dumps(REFERENCIA, ensure_ascii=False))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
for a1, a2, b in F4.divisoes(101):
    assert not (set(a1) & set(b)) and not (set(a2) & set(b)) and not (set(a1) & set(a2))
print("teste com a assinatura oficial | KS conferido contra scipy | dobra externa fora do ajuste e da parada")

# A pilha mede o mesmo que modelo.py na medida nova? O sorteio muda, então cada número muda; o que não pode aparecer é viés.
if GPUS:
    arq = os.path.join(PASTA, "mlp_fase4_conferencia_pilha.json")
    if os.path.isfile(arq):
        conf = json.load(open(arq))["resumo"]; print("Conferência: já concluída, reaproveitada.")
    else:
        t0 = time.time()
        CONFERIDAS = {"referencia": REFERENCIA, "fase3_val": FASE3_VAL, "fase3_teste": FASE3_TESTE}
        tab_conf, conf = F4.conferir_pilha(CONFERIDAS, range(1, 1 + REPS_CONFERENCIA), N_PROCESSOS)
        display(tab_conf.round(4))
        print(f"{(time.time() - t0) / 60:.1f} min")
        salvar_json(os.path.basename(arq), dict(tabela=tab_conf.to_dict("records"), resumo=conf, dispositivos=GPUS, torch=torch.__version__))
    display(pd.Series(conf).to_frame("valor").round(5))
    if not MODO_TESTE:
        assert abs(conf["vies"]) < LIMITE_VIES, f"a pilha mede diferente de modelo.py: {conf}"

# 1. Quanto o protocolo antigo inflava o KS?

Nas mesmas divisões: `val_antigo` é a medida da fase 3 (a mesma dobra decide a parada e mede), `ks_parada` e `ks` são a medida nova. `otimismo_antigo` é quanto o protocolo antigo somava ao número.

In [ ]:
arq = os.path.join(PASTA, "mlp_fase4_protocolos.json")
if os.path.isfile(arq):
    tab_prot = pd.DataFrame(json.load(open(arq))); print("Já concluído, reaproveitado.")
else:
    tab_prot = F4.comparar_protocolos({"referência (fase 2)": REFERENCIA, "campeão val da fase 3 (parada pelo KS)": FASE3_VAL, "campeão teste da fase 3": FASE3_TESTE},
                                      range(601, 601 + REPS_PROTOCOLO), N_PROCESSOS)
    salvar_json(os.path.basename(arq), tab_prot.to_dict("records"))
display(tab_prot.round(4))

# 2. Mapa dos clientes ambíguos

O escore fora da dobra da referência, médio em várias divisões, marca os clientes mais perto do limiar do KS. É entre eles que o KS é decidido. A partir daqui cada medida também guarda o **KS só nesses clientes** (`ks_banda`): uma feature feita para eles precisa melhorar ali.

`relacoes` mostra o que ainda separa quem cancela dentro dessa faixa: o KS de cada variável dentro e fora dela, e as regras de árvores rasas ajustadas em parte da faixa e medidas no resto. `aparece_em` conta em quantas das 5 dobras a mesma regra reaparece. Tudo vem só do treino, então pode ir para os outros modelos do grupo.

In [ ]:
Xtr_, _, ytr_, _ = F4._dados()
y_treino = ytr_.values.astype(int)


def mapa(c, rotulo):
    arq = os.path.join(PASTA, f"mlp_fase4_oof_{rotulo}.npy")
    if os.path.isfile(arq):
        oof = np.load(arq)
    else:
        oof = np.mean([r["oof"] for r in F4.medir_varios([(c, r, 0) for r in range(501, 501 + REPS_MAPA)], N_PROCESSOS)], 0)
        np.save(arq, oof)
    rel = F4.relacoes(Xtr_, y_treino, oof, FRACAO_BANDA)
    print(f"{rotulo}: KS fora da dobra {comum.ks(y_treino, oof):.4f} | {rel['clientes_na_banda']} clientes ambíguos, churn {rel['churn_na_banda']:.3f}, limiar {rel['limiar']:.3f}")
    display(pd.DataFrame(rel["variaveis"]).head(15).round(3))
    display(pd.DataFrame(rel["regras"]).head(15).round(3))
    salvar_json(f"mlp_fase4_relacoes_{rotulo}.json", rel)
    return oof, rel


oof_ini, rel_ini = mapa(REFERENCIA, "inicio")
idx_banda, limiar_banda = F4.banda_ambigua(y_treino, oof_ini, FRACAO_BANDA)
F4.definir_banda(idx_banda)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist([oof_ini[y_treino == 0], oof_ini[y_treino == 1]], bins=40, label=["fica", "cancela"], stacked=True)
ax.axvspan(oof_ini[idx_banda].min(), oof_ini[idx_banda].max(), color="gray", alpha=.25, label="ambíguos")
ax.axvline(limiar_banda, c="k", ls="--"); ax.set_xlabel("escore fora da dobra (referência)"); ax.legend(); plt.show()

# 3. Blocos

Cada bloco parte do campeão anterior e usa as divisões 101 a 130. `delta_parada` é o ganho da rede que parou; `delta_banda`, o ganho nos clientes ambíguos.

In [ ]:
reps_b = range(101, 101 + REPS_BLOCO)
camp = REFERENCIA


def rodar(nome, pergunta, variantes, **k):
    global camp
    if not continuar() or not variantes:
        return None
    camp, t = F4.bloco(PASTA, nome, pergunta, variantes, camp, reps_b, n_jobs=N_PROCESSOS, **k)
    display(t.round(4))
    logar(bloco=nome, ks=float(t.ks.max()))
    backup_periodico()
    return t


PARTIDA = {"campeão val da fase 3, parando pela perda": dict(hp={**FASE3_VAL["hp"], "parada": "perda"}),
           "1x10 com GELU e Dropout": dict(hp=dict(ativ="gelu", plano="D", dropout=.2)),
           "2x16 GELU, Dropout na 1ª camada": dict(hp=dict(camadas=2, unidades=16, ativ="gelu", plano="DN", dropout=.2)),
           "campeão teste da fase 3": dict(hp=FASE3_TESTE["hp"], dados=FASE3_TESTE["dados"])}
rodar("b20_partida", "Os padrões da fase 3 superam a referência na medida nova?", PARTIDA)
rodar("b21_balanceamento", "Trocar as cópias do oversampling por peso na classe positiva ajuda, medindo o retreino?", F4.variantes_balanceamento(camp))

In [ ]:
# Features para os ambíguos: cada rodada testa uma por vez sobre o campeão; a aceita entra e a próxima rodada testa as restantes.
# O XGBoost mede as mesmas variantes de dados nas mesmas dobras: diz se a feature ajuda o dado ou só a rede.
CANDIDATAS_F4 = ["tenure_1", "tenure_ate_3", "novo_mes_a_mes", "n_protecao", "n_streaming", "fibra_sem_protecao", "atrito_pagamento", "sozinho", "sem_gender"]
tabelas_features = []
for rodada in range(1, RODADAS_FEATURES + 1):
    antes = camp
    t = rodar(f"b22_features_r{rodada}", "Alguma feature para os clientes ambíguos supera o ruído?", F4.variantes_features(camp, CANDIDATAS_F4), xgb=True)
    if t is None:
        break
    tabelas_features.append(t.assign(rodada=rodada))
    if camp == antes:
        break
print("Features do campeão:", camp["extras"])

In [ ]:
rodar("b23_treino", "Dropout na 1ª camada, GELU, intensidade, lr, lote ou parada pelo KS (agora na separação interna) ajudam?", F4.variantes_treino(camp))
salvar()
print(json.dumps(camp, ensure_ascii=False, indent=1))

# 4. Optuna na medida nova

Maximiza o KS do retreino no k-fold (divisões 201 e 202). Não há BN no espaço, porque ele perdeu em 1.080 redes na fase 3. O balanceamento entra na busca. O primeiro trial é o campeão. O melhor trial só substitui o campeão se passar pela regra em divisões novas (301 a 330).

In [ ]:
if continuar():
    st, melhor = F4.estudar(PASTA, camp, N_TRIALS, n_jobs=N_PROCESSOS, continuar=continuar, ao_salvar=backup_periodico)
    feitos = st.trials_dataframe()
    feitos = feitos[feitos.state == "COMPLETE"]
    print(f"{len(feitos)} trials | melhor {st.best_value:.4f} | {st.best_params}")
    print(f"Correlação entre o KS do retreino e o da parada nos trials: {feitos.value.corr(feitos.user_attrs_ks_parada):.2f}")
    top = feitos.sort_values("value", ascending=False).head(15)
    for p_ in [c for c in feitos.columns if c.startswith("params_") and (not pd.api.types.is_float_dtype(feitos[c]) or c == "params_camadas")]:
        print(f"  {p_[7:]:<10} top 15: {top[p_].value_counts().to_dict()} | todos: {feitos[p_].value_counts().to_dict()}")
    if melhor != camp:
        camp, t = F4.bloco(PASTA, "b24_optuna", "O melhor trial supera o campeão em divisões novas?", {"melhor trial": dict(hp=melhor["hp"], dados=melhor["dados"])}, camp,
                           range(301, 301 + REPS_BLOCO), n_jobs=N_PROCESSOS)
        display(t.round(4))
salvar()

# 5. Confirmação em divisões que nenhuma escolha viu

Os blocos usam as mesmas divisões em sequência, o que favorece um pouco quem venceu nelas. Aqui o campeão e a referência são medidos em 30 divisões novas (401 a 430): é a estimativa honesta do ganho, ainda sem tocar no teste.

In [ ]:
confirmacao = None
if continuar():
    confirmacao = F4.comparar(REFERENCIA, camp, range(401, 401 + REPS_CONFIRMACAO), N_PROCESSOS)
    display(pd.Series(confirmacao).to_frame("valor"))
    salvar_json("mlp_fase4_confirmacao.json", dict(confirmacao, referencia=REFERENCIA, campeao=camp))
if continuar() and camp != REFERENCIA:
    oof_fim, rel_fim = mapa(camp, "fim")
    print(f"Nos mesmos clientes ambíguos: KS da referência {comum.ks(y_treino[idx_banda], oof_ini[idx_banda]):.4f} | do campeão {comum.ks(y_treino[idx_banda], oof_fim[idx_banda]):.4f}")

# 6. Avaliação final: o teste, uma vez só

Treino inteiro, 5 sementes, limiar vindo da validação, como nas fases anteriores. Nada depois desta célula muda a escolha.

In [ ]:
finais = {}
if continuar():
    for rot, c in (("referencia", REFERENCIA), ("campeao", camp)):
        if rot == "campeao" and c == REFERENCIA:
            finais[rot] = finais["referencia"]; continue
        finais[rot] = BL.final_campeao(c, N_SEMENTES)
        m = finais[rot]["teste"]
        print(f"{rot:<11} KS = {m['ks']:.4f} | IC 95% {np.round(m['ks_ic95'], 4)} | AUROC {m['auroc']:.4f} | por semente {np.round(finais[rot]['ks_teste_por_semente'], 4)}")
    y_f = F4._dados()[3].values.astype("float32")
    delta_teste = F3.bootstrap_delta(y_f, np.array(finais["campeao"]["probabilidades_teste"]), np.array(finais["referencia"]["probabilidades_teste"]))
    print("campeão - referência no teste:", delta_teste)
    salvar_json("mlp_fase4_final.json", dict(finais=finais, campeao_menos_referencia=delta_teste, confirmacao_kfold=confirmacao))

In [ ]:
# Em quem o campeão erra no teste e o custo de trocar um erro pelo outro. Só mostra: nada aqui volta para o treino.
if "campeao" in finais:
    grupos, um, pares = F3.erros_no_teste(finais["campeao"]["probabilidades_teste"])
    troca = F3.trade_off_erros(finais["campeao"]["probabilidades_teste"], finais["campeao"]["teste"]["limiar"])
    print(f"Resíduos com q < 0,05: {int((um.q_bh < .05).sum())} de {len(um)} grupos, {int((pares.q_bh < .05).sum())} de {len(pares)} pares")
    display(grupos.head(10).round(3)); display(troca.round(4))
    salvar_json("mlp_fase4_erros_teste.json", dict(por_grupo=grupos.to_dict("records"), residuos=um.to_dict("records"), pares=pares.head(100).to_dict("records"), trade_off=troca.to_dict("records")))

### 📝 Análise da fase 4

- **Quanto o protocolo antigo inflava o KS?** _…_
- **Os padrões da fase 3 se sustentam na medida nova?** _…_
- **Oversampling ou peso?** _…_
- **Alguma feature separou os ambíguos? Vale também no XGBoost?** _…_
- **Ganho confirmado em divisões novas e no teste:** _…_
- **Próxima hipótese:** _…_

In [ ]:
# O que vai para os outros modelos do grupo: features testadas no MLP e no XGBoost e as relações entre os ambíguos
para_o_grupo = dict(aviso="Medidas no k-fold do treino (divisões 101 a 130); o teste não entrou. delta_xgb usa o XGBoost de erros._xgb.",
                    features=pd.concat(tabelas_features).to_dict("records") if tabelas_features else [], relacoes_inicio=rel_ini,
                    campeao=camp, confirmacao=confirmacao)
salvar_json("mlp_fase4_para_o_grupo.json", para_o_grupo)
salvar_json("mlp_fase4_campeao.json", camp)
if wandb_run is not None:
    wandb_run.summary.update(dict(campeao=json.dumps(camp), confirmacao=json.dumps(confirmacao)))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))